In [ ]:
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

KAFKA_BOOTSTRAP_SERVERS = "urbangreen-kafka:9092"
KAFKA_TOPIC = "sensor_readings"

MINIO_ENDPOINT = "http://urbangreen-minio:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
MINIO_BUCKET = os.getenv("MINIO_STAGING_BUCKET", "staging")

CLICKHOUSE_HOST = os.getenv(
    "CLICKHOUSE_HOST",
    "urbangreen-clickhouse",
)
CLICKHOUSE_HTTP_PORT = int(
    os.getenv("CLICKHOUSE_HTTP_PORT", "8123")
)
CLICKHOUSE_DATABASE = (
    os.getenv("CLICKHOUSE_DB_NAME")
    or os.getenv("CLICKHOUSE_DB")
    or "urbangreen_dw"
)
CLICKHOUSE_USER = os.getenv(
    "CLICKHOUSE_USER"
)
CLICKHOUSE_PASSWORD = os.getenv(
    "CLICKHOUSE_PASSWORD"
)
CLICKHOUSE_JDBC_URL = (
    f"jdbc:clickhouse://{CLICKHOUSE_HOST}:"
    f"{CLICKHOUSE_HTTP_PORT}/{CLICKHOUSE_DATABASE}"
)
CLICKHOUSE_JDBC_DRIVER = (
    "com.clickhouse.jdbc.ClickHouseDriver"
)
CLICKHOUSE_JDBC_JAR = (
    "https://repo1.maven.org/maven2/"
    "com/clickhouse/clickhouse-jdbc/0.9.8/"
    "clickhouse-jdbc-0.9.8-all-dependencies.jar"
)

spark = (
    SparkSession.builder
    .appName("sensor-readings-stream-notebook-test")
    .master("spark://urbangreen-spark-master:7077")
    .config("spark.driver.host", "urbangreen-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")

    # Kafka connector
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
            "org.apache.hadoop:hadoop-aws:3.4.1",
        ])
    )

    # Exact ClickHouse uber-JAR required by the ticket.
    .config(
        "spark.jars",
        CLICKHOUSE_JDBC_JAR,
    )

    # MinIO/S3A config
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )

    # Keep all timestamp transformations in UTC.
    .config(
        "spark.sql.session.timeZone",
        "UTC",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_ENDPOINT)
print("Bucket:", MINIO_BUCKET)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a38c6fb8-e688-41c9-9a25-f2d9d2a7dbc8;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.cod

Spark version: 4.0.2
MinIO endpoint: http://urbangreen-minio:9000
Bucket: staging


In [4]:
sensor_df = spark.read.parquet(
    "s3a://staging/raw/kafka/sensor_readings/"
)
sensor_df.printSchema()
sensor_df.show(5, truncate=False)

root
 |-- farm_sensor_id: integer (nullable = true)
 |-- farm_id: integer (nullable = true)
 |-- sensor_type_id: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- event_date: date (nullable = true)

+--------------+-------+--------------+------+----------+----------+
|farm_sensor_id|farm_id|sensor_type_id|value |timestamp |event_date|
+--------------+-------+--------------+------+----------+----------+
|1             |1      |1             |22.122|1784706731|2026-07-22|
|5             |1      |5             |66.126|1784706731|2026-07-22|
|7             |2      |1             |22.583|1784706731|2026-07-22|
|8             |2      |2             |62.695|1784706731|2026-07-22|
|11            |2      |5             |63.766|1784706731|2026-07-22|
+--------------+-------+--------------+------+----------+----------+
only showing top 5 rows


26/07/22 20:54:03 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/07/22 20:54:03 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce

In [2]:
crops_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("s3a://staging/raw/postgres/harvests/")
)

crops_df.printSchema()
crops_df.show(5, truncate=False)

26/07/22 12:01:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- id: long (nullable = true)
 |-- farm_id: long (nullable = true)
 |-- crop_id: long (nullable = true)
 |-- quality_grade_id: long (nullable = true)
 |-- weight_kg: decimal(4,3) (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)

+------+-------+-------+----------------+---------+----------+----------+
|id    |farm_id|crop_id|quality_grade_id|weight_kg|created_at|updated_at|
+------+-------+-------+----------------+---------+----------+----------+
|600716|44     |6      |3               |0.797    |1768587540|1768587540|
|601446|44     |7      |1               |1.539    |1768568520|1768568520|
|602176|44     |8      |1               |1.582    |1768587780|1768587780|
|602906|44     |9      |2               |1.439    |1768526400|1768526400|
|603636|44     |10     |1               |1.316    |1768547520|1768547520|
+------+-------+-------+----------------+---------+----------+----------+
only showing top 5 rows


26/07/22 12:04:21 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/07/22 12:04:21 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce

In [4]:
from datetime import datetime, timezone

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

In [6]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

26/07/22 11:47:00 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/07/22 11:47:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce

In [6]:
MINIO_BUCKET = "staging"

CROPS_PATH = (
    f"s3a://{MINIO_BUCKET}/raw/postgres/crops/"
)

CROP_CATEGORIES_PATH = (
    f"s3a://{MINIO_BUCKET}/raw/postgres/crop_categories/"
)

DIM_CROP_TEST_PATH = (
    f"s3a://{MINIO_BUCKET}/test/warehouse/dim_crop/"
)

print("Crops source:", CROPS_PATH)
print("Categories source:", CROP_CATEGORIES_PATH)
print("Test output:", DIM_CROP_TEST_PATH)

Crops source: s3a://staging/raw/postgres/crops/
Categories source: s3a://staging/raw/postgres/crop_categories/
Test output: s3a://staging/test/warehouse/dim_crop/


In [7]:
def read_postgres_history(
    table_path: str,
) -> DataFrame:
    """
    Read all Parquet files from all extraction batches.

    The source file path is retained as technical metadata so
    it can also serve as a deterministic tie-breaker.
    """

    return (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(table_path)
        .withColumn(
            "_source_file",
            F.input_file_name(),
        )
    )

In [8]:
crops_raw_df = read_postgres_history(CROPS_PATH)

crops_raw_df.printSchema()

crops_raw_df.show(
    10,
    truncate=False,
)

root
 |-- id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)
 |-- _source_file: string (nullable = false)

+---+-----------+------------------+------------------------------------------------------------------------------------+----------+----------+---------------------------------------------------------------------------------+
|id |category_id|name              |description                                                                         |created_at|updated_at|_source_file                                                                     |
+---+-----------+------------------+------------------------------------------------------------------------------------+----------+----------+---------------------------------------------------------------------------------+
|1  |2          |Basil             |Aro

In [9]:
crop_categories_raw_df = read_postgres_history(
    CROP_CATEGORIES_PATH
)

crop_categories_raw_df.printSchema()

crop_categories_raw_df.show(
    20,
    truncate=False,
)

print(
    "Raw crop category rows:",
    crop_categories_raw_df.count(),
)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)
 |-- _source_file: string (nullable = false)

+---+---------------+------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+-----------------------------------------------------------------------------------------------------+
|id |name           |description                                                                                                                               |created_at|updated_at|_source_file                                                                                         |
+---+---------------+------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+------

In [10]:
def require_columns(
    dataframe: DataFrame,
    required_columns: list[str],
    source_name: str,
) -> None:
    missing_columns = sorted(
        set(required_columns) - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{source_name} is missing required columns: "
            f"{missing_columns}. "
            f"Available columns: "
            f"{sorted(dataframe.columns)}"
        )

In [11]:
require_columns(
    crops_raw_df,
    [
        "id",
        "category_id",
        "name",
        "description",
        "created_at",
        "updated_at",
        "_source_file",
    ],
    "crops",
)

In [12]:
require_columns(
    crop_categories_raw_df,
    [
        "id",
        "name",
        "created_at",
        "updated_at",
        "_source_file",
    ],
    "crop_categories",
)

In [13]:
def latest_row_per_key(
    dataframe: DataFrame,
    key_column: str,
) -> DataFrame:
    """
    Keep exactly one latest source row per natural key.

    Ordering:
    1. greatest updated_at
    2. greatest created_at
    3. source file path as deterministic tie-breaker
    """

    window = (
        Window
        .partitionBy(key_column)
        .orderBy(
            F.col("updated_at").desc_nulls_last(),
            F.col("created_at").desc_nulls_last(),
            F.col("_source_file").desc(),
        )
    )

    return (
        dataframe
        .withColumn(
            "_row_number",
            F.row_number().over(window),
        )
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

In [14]:
crops_latest_df = latest_row_per_key(
    crops_raw_df,
    key_column="id",
)

crop_categories_latest_df = latest_row_per_key(
    crop_categories_raw_df,
    key_column="id",
)

In [15]:
categories_for_join_df = (
    crop_categories_latest_df
    .select(
        F.col("id")
        .cast("long")
        .alias("crop_category_id"),

        F.trim(
            F.col("name")
        ).alias("category_name"),
    )
)

In [16]:
categories_for_join_df.orderBy(
    "crop_category_id"
).show(
    truncate=False
)

+----------------+---------------+
|crop_category_id|category_name  |
+----------------+---------------+
|1               |Leafy Greens   |
|2               |Herbs          |
|3               |Microgreens    |
|4               |Specialty Crops|
+----------------+---------------+



In [17]:
crops_with_category_df = (
    crops_latest_df.alias("crop")
    .join(
        categories_for_join_df.alias("category"),
        F.col("crop.category_id")
        == F.col("category.crop_category_id"),
        "left",
    )
)

In [18]:
crops_with_category_df.select(
    F.col("crop.id").alias("crop_id"),
    F.col("crop.name").alias("crop_name"),
    F.col("crop.category_id"),
    F.col("category.category_name"),
).orderBy(
    "crop_id"
).show(
    truncate=False
)

+-------+------------------+-----------+---------------+
|crop_id|crop_name         |category_id|category_name  |
+-------+------------------+-----------+---------------+
|1      |Basil             |2          |Herbs          |
|2      |Mint              |2          |Herbs          |
|3      |Parsley           |2          |Herbs          |
|4      |Cilantro          |2          |Herbs          |
|5      |Thyme             |2          |Herbs          |
|6      |Rosemary          |2          |Herbs          |
|7      |Romaine Lettuce   |1          |Leafy Greens   |
|8      |Butterhead Lettuce|1          |Leafy Greens   |
|9      |Arugula Lettuce   |1          |Leafy Greens   |
|10     |Spinach           |1          |Leafy Greens   |
|11     |Kale              |1          |Leafy Greens   |
|12     |Swiss Chard       |1          |Leafy Greens   |
|13     |Radish            |3          |Microgreens    |
|14     |Pea Shoots        |3          |Microgreens    |
|15     |Sunflower         |3  

In [19]:
load_timestamp = datetime.now(
    timezone.utc
).replace(
    tzinfo=None,
)

In [20]:
load_timestamp = load_timestamp.replace(
    microsecond=(
        load_timestamp.microsecond // 1000
    ) * 1000
)

print("Loader timestamp:", load_timestamp)

Loader timestamp: 2026-07-21 12:00:19.352000


In [21]:
dim_crop_df = (
    crops_with_category_df
    .select(
        F.col("crop.id")
        .cast("long")
        .alias("crop_id"),

        F.trim(
            F.col("crop.name")
        ).cast("string").alias("name"),

        F.coalesce(
            F.trim(
                F.col("crop.description")
            ),
            F.lit(""),
        ).cast("string").alias("description"),

        F.col("crop.category_id")
        .cast("long")
        .alias("crop_category_id"),

        F.trim(
            F.col("category.category_name")
        ).cast("string").alias("category_name"),

        F.lit(load_timestamp)
        .cast("timestamp")
        .alias("_loaded_at"),
    )
)

In [22]:
dim_crop_df.printSchema()

root
 |-- crop_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = false)
 |-- crop_category_id: long (nullable = true)
 |-- category_name: string (nullable = true)
 |-- _loaded_at: timestamp (nullable = false)



In [23]:
dim_crop_df.orderBy(
    "crop_id"
).show(
    50,
    truncate=False,
)

+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|crop_id|name              |description                                                                                                    |crop_category_id|category_name  |_loaded_at             |
+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|1      |Basil             |Aromatic herb used in Mediterranean cuisine                                                                    |2               |Herbs          |2026-07-21 12:00:19.352|
|2      |Mint              |Refreshing herb used in drinks and culinary dishes                                                             |2               |Herbs          |2026-07-21 12:00:19.352|
|3      |P

In [24]:
(
    dim_crop_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(DIM_CROP_TEST_PATH)
)

In [25]:
print(
    f"dim_crop successfully written to "
    f"{DIM_CROP_TEST_PATH}"
)

dim_crop successfully written to s3a://staging/test/warehouse/dim_crop/


In [26]:
print(

    spark.sparkContext._jsc.sc().listJars().toString()

)

List(https://repo1.maven.org/maven2/com/clickhouse/clickhouse-jdbc/0.9.8/clickhouse-jdbc-0.9.8-all-dependencies.jar)


In [27]:
import os

CLICKHOUSE_HOST = os.getenv(
    "CLICKHOUSE_HOST",
    "urbangreen-clickhouse",
)

CLICKHOUSE_HTTP_PORT = int(
    os.getenv("CLICKHOUSE_HTTP_PORT", "8123")
)

CLICKHOUSE_DB = (
    os.getenv("CLICKHOUSE_DB_NAME")
    or os.getenv("CLICKHOUSE_DB")
    or "urbangreen_dw"
)

CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD")

CLICKHOUSE_JDBC_URL = (
    f"jdbc:clickhouse://{CLICKHOUSE_HOST}:"
    f"{CLICKHOUSE_HTTP_PORT}/{CLICKHOUSE_DB}"
)

CLICKHOUSE_TABLE = f"{CLICKHOUSE_DB}.dim_crop"

print("JDBC URL:", CLICKHOUSE_JDBC_URL)
print("Target table:", CLICKHOUSE_TABLE)

JDBC URL: jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw
Target table: urbangreen_dw.dim_crop


In [28]:
from pathlib import Path
import pyspark

print(Path(pyspark.__file__).parent / "jars")

/opt/conda/lib/python3.11/site-packages/pyspark/jars


In [29]:
dim_crop_df.printSchema()

dim_crop_df.orderBy("crop_id").show(
    truncate=False
)

print("Rows to write:", dim_crop_df.count())

root
 |-- crop_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = false)
 |-- crop_category_id: long (nullable = true)
 |-- category_name: string (nullable = true)
 |-- _loaded_at: timestamp (nullable = false)

+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|crop_id|name              |description                                                                                                    |crop_category_id|category_name  |_loaded_at             |
+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|1      |Basil             |Aromatic herb used in Mediterranean cuisine                                                                    |2     

In [30]:
dim_crop_to_write_df = dim_crop_df.select(
    "crop_id",
    "name",
    "description",
    "crop_category_id",
    "category_name",
    "_loaded_at",
)

In [31]:
(
    dim_crop_to_write_df
    .coalesce(1)
    .write
    .format("jdbc")
    .option("url", CLICKHOUSE_JDBC_URL)
    .option("dbtable", CLICKHOUSE_TABLE)
    .option(
        "driver",
        "com.clickhouse.jdbc.ClickHouseDriver",
    )
    .option("user", CLICKHOUSE_USER)
    .option("password", CLICKHOUSE_PASSWORD)
    .option("batchsize", "1000")
    .mode("append")
    .save()
)

print(
    f"Successfully wrote dim_crop to "
    f"{CLICKHOUSE_TABLE}."
)

Successfully wrote dim_crop to urbangreen_dw.dim_crop.


In [32]:
saved_dim_crop_df = (
    spark.read
    .format("jdbc")
    .option("url", CLICKHOUSE_JDBC_URL)
    .option(
        "query",
        f"""
        SELECT
            crop_id,
            name,
            description,
            crop_category_id,
            category_name,
            _loaded_at
        FROM {CLICKHOUSE_TABLE}
        FINAL
        ORDER BY crop_id
        """,
    )
    .option(
        "driver",
        "com.clickhouse.jdbc.ClickHouseDriver",
    )
    .option("user", CLICKHOUSE_USER)
    .option("password", CLICKHOUSE_PASSWORD)
    .load()
)

In [33]:
saved_dim_crop_df.printSchema()

saved_dim_crop_df.show(
    50,
    truncate=False,
)

root
 |-- crop_id: decimal(20,0) (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- crop_category_id: decimal(20,0) (nullable = true)
 |-- category_name: string (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|crop_id|name              |description                                                                                                    |crop_category_id|category_name  |_loaded_at             |
+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+-----------------------+
|1      |Basil             |Aromatic herb used in Mediterranean cuisine                                                           

26/07/21 12:16:15 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/07/21 12:16:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce